# k05 — Cross-surface: open a CLI session from Python

**The claim under test** (the point of the whole knowledge-agent
initiative): the `finstack-know` CLI and these notebooks share one
journaled sqlite session store — a session created on one surface opens
on the other, with full history, and the conversation continues.

Trust: T2 callback (offline path) / T1 native provider (live path).
Network: none by default.

## Create the "CLI" session

The real command is:

```bash
finstack-know --data-dir <data_dir> ask "What are the six runtime ports?"
# session: 01a0...            <- printed id
```

Run it yourself when you have the binary (`cargo build -p
finstack-ai-knowledge --bin finstack-know`) and a local Ollama. This
notebook must execute offline and deterministically, so the next cell is
the **documented stand-in**: it produces exactly what the CLI produces —
a session in `<data_dir>/journal.sqlite3` with tenant scope `local` and
one completed turn on lane `main` — using a scripted model instead of a
live provider. Everything downstream (open, inspect, continue) is
identical either way.

In [1]:
import tempfile
from pathlib import Path

from _knowledge import build_knowledge_agent, scripted_model

workdir = Path(tempfile.mkdtemp(prefix="finstack-know-k05-"))

cli_like = await build_knowledge_agent(
    workdir,
    scripted_model(
        [
            "The six runtime ports are model, tool, context, middleware, observer, and journal."
        ],
        component="knowledge.model.k05-cli-standin",
    ),
    tenant="local",  # the CLI's tenant scope
)
cli_session = await cli_like.create_session("local")
cli_lane = await cli_session.lane("main")
first = await cli_lane.run(cli_like, "What are the six runtime ports?").result()
session_id = cli_session.session_id
print("session:", session_id)
print(first.text)

session: 01a04f6b-4dd0-726c-9e92-7ed339789a72
The six runtime ports are model, tool, context, middleware, observer, and journal.


## Open it from a second composition and inspect

A fresh agent (a stand-in for "a different process on a different day")
opens the same journal file and the same session id with the same tenant
scope. The lane history is all there — both entries of the first
turn.

In [2]:
inspector = await build_knowledge_agent(
    workdir,
    scripted_model(
        [
            "Sessions and lanes live in the journal; every surface reads the same records."
        ],
        component="knowledge.model.k05-inspector",
    ),
    tenant="local",
)
reopened = await inspector.open_session(session_id, "local")
lane = await reopened.lane("main")
info = await lane.inspect()
print("lane:", info["name"], "history entries:", info["history_len"])
assert info["name"] == "main"
assert info["history_len"] >= 2, "the CLI turn is visible from Python"

lane: main history entries: 2


## Continue the conversation from Python

`lane.run` appends the next turn to the same durable history — Python is
now mid-conversation in a session the CLI started.

In [3]:
followup = await lane.run(inspector, "And where do sessions live?").result()
print(followup.text)

info_after = await (await reopened.lane("main")).inspect()
print("history entries now:", info_after["history_len"])
assert info_after["history_len"] > info["history_len"]

Sessions and lanes live in the journal; every surface reads the same records.


history entries now: 4


## Hand it back to the CLI

Point the CLI at the same data directory and the session shows both
turns — the one created here and the one created "there":

```bash
finstack-know --data-dir <data_dir> sessions list
finstack-know --data-dir <data_dir> sessions show <session-id>
finstack-know --data-dir <data_dir> ask "continue" --session <session-id>
```

`sessions show` renders the same lane history `lane.inspect()` returned
above: user and assistant turns in order, from one shared journal. That
round trip — CLI → Python → CLI, one sqlite file, no export step — is
the initiative's central claim, demonstrated.

In [4]:
import shutil

print("data dir to point the CLI at:", workdir)
print("session id:", session_id)

# Keep the directory if you want to try the CLI commands above;
# otherwise clean up:
shutil.rmtree(workdir, ignore_errors=True)
print("cleaned", workdir)

data dir to point the CLI at: /var/folders/l1/s1m3_kfn43d77mc45c_3rv_h0000gn/T/finstack-know-k05-b5_166yr
session id: 01a04f6b-4dd0-726c-9e92-7ed339789a72
cleaned /var/folders/l1/s1m3_kfn43d77mc45c_3rv_h0000gn/T/finstack-know-k05-b5_166yr
